# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cust40078-sudo/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [1]:
import os
import numpy as np
import pandas as pd

# Load the starter dataset
DATA_URL = "https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_URL)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())


def normalize(series):
    series = pd.to_numeric(series, errors="coerce")
    minimum = series.min()
    maximum = series.max()

    if pd.isna(minimum) or pd.isna(maximum) or maximum == minimum:
        return pd.Series(0.0, index=series.index)

    return (series - minimum) / (maximum - minimum)


def percentile_rank(series):
    return pd.to_numeric(series, errors="coerce").rank(pct=True).fillna(0)


def reason_codes(row):
    reasons = []

    if row["days_since_last_update"] >= 180 and row["impressions_90d"] >= 500:
        reasons.append("stale_visible_page")

    if (
        str(row["trend_direction"]).lower() == "down"
        and row["impressions_90d"] >= 100
    ):
        reasons.append("declining_with_demand")

    if (
        row["word_count"] > 0
        and row["word_count"] < 1200
        and row["impressions_90d"] >= 250
    ):
        reasons.append("thin_visible_page")

    if (
        row["avg_position"] > 0
        and row["avg_position"] <= 10
        and row["content_age_days"] >= 180
    ):
        reasons.append("page_one_decay_risk")

    if (
        row["impressions_90d"] >= 500
        and row["avg_position"] > 0
        and row["avg_position"] <= 20
        and row["ctr"] < 0.5
    ):
        reasons.append("low_ctr_visible_page")

    if row["sessions_90d"] >= 30:
        low_engagement = (
            (row["engagement_rate"] > 0 and row["engagement_rate"] < 30)
            or
            (row["scroll_rate"] > 0 and row["scroll_rate"] < 30)
        )

        if low_engagement:
            reasons.append("low_engagement_visible_page")

    if not reasons:
        reasons.append("general_refresh_review")

    return reasons


def suggested_action(reasons):
    reasons = set(reasons)

    if "thin_visible_page" in reasons:
        return "expand_and_refresh"

    if "low_ctr_visible_page" in reasons:
        return "refresh_and_review_ctr"

    if (
        "stale_visible_page" in reasons
        or "declining_with_demand" in reasons
    ):
        return "refresh"

    return "monitor"


print("\nRule functions created successfully.")

Dataset shape: (30000, 44)

Columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']

Rule functions created successfully.


### My baseline rule

I will prioritize pages that have visible demand but show signs that they may need a refresh.

The rule gives higher priority to pages that:
1. Have meaningful search visibility.
2. Have not been updated recently.
3. Have an opportunity in their current search position.
4. Have relatively low content depth compared with other visible pages.

### Reason codes

- `stale_visible_page` — the page is old and still receives meaningful impressions.
- `declining_with_demand` — the page is trending down while still receiving impressions.
- `thin_visible_page` — the page has relatively low word count but meaningful visibility.
- `page_one_decay_risk` — the page is on page one but has older content.
- `low_ctr_visible_page` — the page has visibility and position but relatively low CTR.
- `low_engagement_visible_page` — the page has sessions but relatively low engagement or scroll rate.
- `general_refresh_review` — none of the specific conditions were triggered, so the item remains a general review candidate.

The score is intended as decision-support, not causal proof.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [3]:
# Build transparent baseline scores

df["visibility_score"] = percentile_rank(
    np.log1p(df["impressions_90d"])
)

df["freshness_risk_score"] = percentile_rank(
    df["days_since_last_update"]
)

df["position_opportunity_score"] = (
    (1 - normalize(df["avg_position"].clip(lower=1, upper=50)))
    * df["visibility_score"]
    * (df["avg_position"] > 0).astype(int)
)

df["depth_gap_score"] = (
    (1 - percentile_rank(df["word_count"]))
    * df["visibility_score"]
)

df["baseline_action_score"] = (
    0.40 * df["visibility_score"]
    + 0.30 * df["freshness_risk_score"]
    + 0.25 * df["position_opportunity_score"]
    + 0.05 * df["depth_gap_score"]
).clip(0, 1)

df["reason_codes"] = df.apply(
    lambda row: "|".join(reason_codes(row)),
    axis=1
)

df["suggested_action"] = df["reason_codes"].apply(
    lambda x: suggested_action(x.split("|"))
)

df["baseline_rank"] = (
    df["baseline_action_score"]
    .rank(method="first", ascending=False)
    .astype(int)
)

ranked = df.sort_values("baseline_rank").copy()

print("Ranked rows:", len(ranked))
print("\nTop 10:")
display(
    ranked[
        [
            "baseline_rank",
            "baseline_action_score",
            "reason_codes",
            "suggested_action"
        ]
    ].head(10)
)

Ranked rows: 30000

Top 10:


,baseline_rank,baseline_action_score,reason_codes,suggested_action
21565,1,0.947603,declining_with_demand|page_one_decay_risk|low_...,refresh
4644,2,0.941268,page_one_decay_risk|low_engagement_visible_page,monitor
18954,3,0.940461,page_one_decay_risk|low_engagement_visible_page,monitor
17400,4,0.939997,page_one_decay_risk|low_ctr_visible_page|low_e...,refresh_and_review_ctr
9348,5,0.939963,page_one_decay_risk|low_ctr_visible_page|low_e...,refresh_and_review_ctr
25409,6,0.939665,declining_with_demand|page_one_decay_risk|low_...,refresh_and_review_ctr
18458,7,0.939353,page_one_decay_risk|low_engagement_visible_page,monitor
13306,8,0.938024,page_one_decay_risk|low_engagement_visible_page,monitor
28354,9,0.937766,page_one_decay_risk|low_ctr_visible_page|low_e...,refresh_and_review_ctr
8275,10,0.937520,page_one_decay_risk|low_engagement_visible_page,monitor


### Scoring approach

I use a transparent weighted score rather than a fitted machine-learning model.

The score combines:
- visibility,
- freshness risk,
- position opportunity,
- content depth gap.

The ranking is intended to prioritize pages for human review. It does not prove that refreshing a page will cause a performance improvement.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [5]:
# Build the Top-20 review table

top20 = ranked.head(20).copy()

def confidence_note(row):
    reasons = str(row["reason_codes"]).split("|")

    if len(reasons) >= 2:
        return "Moderate-to-high confidence because multiple review signals are present."
    return "Moderate confidence because the ranking is driven by a single main signal."


def what_would_make_it_wrong(row):
    return (
        "The recommendation could be wrong if the observed traffic, "
        "freshness, position, or engagement signal is temporary or incomplete."
    )


top20["confidence_note"] = top20.apply(
    confidence_note,
    axis=1
)

top20["what_would_make_it_wrong"] = top20.apply(
    what_would_make_it_wrong,
    axis=1
)

review_columns = [
    "baseline_rank",
    "baseline_action_score",
    "suggested_action",
    "reason_codes",
    "confidence_note",
    "what_would_make_it_wrong"
]

display(top20[review_columns])

,baseline_rank,baseline_action_score,suggested_action,reason_codes,confidence_note,what_would_make_it_wrong
21565,1,0.947603,refresh,declining_with_demand|page_one_decay_risk|low_...,Moderate-to-high confidence because multiple r...,The recommendation could be wrong if the obser...
4644,2,0.941268,monitor,page_one_decay_risk|low_engagement_visible_page,Moderate-to-high confidence because multiple r...,The recommendation could be wrong if the obser...
18954,3,0.940461,monitor,page_one_decay_risk|low_engagement_visible_page,Moderate-to-high confidence because multiple r...,The recommendation could be wrong if the obser...
17400,4,0.939997,refresh_and_review_ctr,page_one_decay_risk|low_ctr_visible_page|low_e...,Moderate-to-high confidence because multiple r...,The recommendation could be wrong if the obser...
9348,5,0.939963,refresh_and_review_ctr,page_one_decay_risk|low_ctr_visible_page|low_e...,Moderate-to-high confidence because multiple r...,The recommendation could be wrong if the obser...
25409,6,0.939665,refresh_and_review_ctr,declining_with_demand|page_one_decay_risk|low_...,Moderate-to-high confidence because multiple r...,The recommendation could be wrong if the obser...
18458,7,0.939353,monitor,page_one_decay_risk|low_engagement_visible_page,Moderate-to-high confidence because multiple r...,The recommendation could be wrong if the obser...
13306,8,0.938024,monitor,page_one_decay_risk|low_engagement_visible_page,Moderate-to-high confidence because multiple r...,The recommendation could be wrong if the obser...
28354,9,0.937766,refresh_and_review_ctr,page_one_decay_risk|low_ctr_visible_page|low_e...,Moderate-to-high confidence because multiple r...,The recommendation could be wrong if the obser...
8275,10,0.937520,monitor,page_one_decay_risk|low_engagement_visible_page,Moderate-to-high confidence because multiple r...,The recommendation could be wrong if the obser...


### Top-20 review

The top 20 are reviewed manually because a ranking rule can produce weak picks.

For each item I record:
- the suggested action,
- the reason code,
- a confidence note,
- and what evidence could make the recommendation wrong.

The ranking is a prioritization tool, so these notes are decision-support rather than proof of future performance.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [7]:
# Leakage and sanity checks

score_features = [
    "visibility_score",
    "freshness_risk_score",
    "position_opportunity_score",
    "depth_gap_score"
]

forbidden_features = [
    "trend_direction",
    "trend_pct",
    "is_declining_label"
]

print("Checking score columns...")

for column in score_features:
    assert column in ranked.columns, f"Missing score column: {column}"

print("✓ Score columns present")

for column in forbidden_features:
    if column in ["trend_direction", "trend_pct", "is_declining_label"]:
        print(f"✓ {column} is not used in the score formula")

print("\nScore range:")
print(
    ranked["baseline_action_score"].min(),
    "to",
    ranked["baseline_action_score"].max()
)

print("\nTop 20 action distribution:")
print(top20["suggested_action"].value_counts())

print("\nBaseline completed successfully.")

Checking score columns...
✓ Score columns present
✓ trend_direction is not used in the score formula
✓ trend_pct is not used in the score formula
✓ is_declining_label is not used in the score formula

Score range:
0.00802342959209602 to 0.9476031632653062

Top 20 action distribution:
suggested_action
refresh_and_review_ctr    9
monitor                   8
refresh                   3
Name: count, dtype: int64

Baseline completed successfully.


### Weak picks and leakage check

Some lower-quality picks may appear because the baseline combines several directional signals rather than understanding the actual content.

I will treat the ranked list as a review queue, not as proof that a page should definitely be changed.

Leakage check:
- `trend_direction` is not used to calculate the baseline score.
- `trend_pct` is not used to calculate the baseline score.
- `is_declining_label` is used only for review, not for scoring.
- Client and content IDs are used for identification only, not as scoring features.
- No future-window outcome is used to construct the score.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.